In [1]:
#TODO make sure this renders in the github repo

# Training an LLM

🌟 **LLM training occurs in two phases:**
1. **Pre-Training** (Phase 1):
   - Objective: (Teacher Forcing) Next-Token prediction over a massive, unstructured dataset (trillions of words from the internet, books, code).
   - Goal: To teach the model the structure of human language, logic, coding syntax, and general world knowledge. This is the base model.
   - Steps: 
      1. Pass a input sequence of tokens through the model.
         - The tokens are masked so the model can not see future tokens, and specific to Llama, a document mask that prevents a token from one document from attending to tokens in other documents.
         - The ground truth is the same as the input, but shifted right by one token.
      2. The models outputs a prediction for every position at once, and we calculate cross-entropy loss between the model's prediction and the ground truth.

2. [**Post-Training** (Phase 2)](./post-training.ipynb)


**Llama 3 paper:**
- **Pre-training:**
  - "We start by converting a large, multilingual text corpus to discrete tokens and pre-training a large language model (LLM) on the resulting data to perform next-token prediction. In the language model pre-training stage, the model learns the structure of language and obtains large amounts of knowledge about the world from the text it is “reading”. To do this effectively, pre-training is performed at massive scale: we pre-train a model with 405B parameters on 15.6T tokens using a context window of 8K tokens. This standard pre-training stage is followed by a continued pre-training stage that increases the supported context window to 128K tokens. See Section 3 for details." 
  - "During pre-training on the final 40M tokens, we linearly annealed the learning rate to 0, maintaining a context length of 128K tokens. During this annealing phase, we also adjusted the data mix to upsample data sources of very high quality; see Section 3.1.3. Finally, we compute the average of model checkpoints (Polyak (1991) averaging) during annealing to produce the final pre-trained model."



**Notes:**
- Instead of training on a **number of epochs**, the model is trained on a **number of steps**. Once it reaches the **total_training_steps** (e.g., 1,200,000), training ends. This is done because the datasets for LLM training are so massive, that the model will rarely see the same piece of text twice. Training per epoch just means that for every epoch, the model sees the same dataset.
- The dataloader yields dense, packed batches of tokens.

In [2]:
import EasyJupyter
import torch
from torch.utils.data import DataLoader
import torch.nn as nn

In [3]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from llama_configs import BaseConfig
    from model.decoder import Decoder

## Loss Function

**Llama 3 Paper:** 
- "Language model **pre-training**. **We start by converting a large, multilingual text corpus to discrete tokens and pre-training a large language model (LLM) on the resulting data to perform next-token prediction.** **In the language model pre-training stage, the model learns the structure of language and obtains large amounts of knowledge about the world from the text it is “reading”.** To do this effectively, pre-training is performed at massive scale: we pre-train a model with 405B parameters on 15.6T tokens using a context window of 8K tokens. This standard pre-training stage is followed by a continued pre-training stage that increases the supported context window to 128K tokens. See Section 3 for details."


**Notes:**
- **Cross-Entropy Loss** was used for the **Pre-Training** (to calculate the error of next-token prediction compared to the ground truth) and in the SFT portion of **Post-Training**. **DPO Loss** is used in the alignment/Direct Preference Optimization portion of **Post-Training**.

## Train Model

[Optimizer and Scheduler](./utils/optimizer_and_scheduler.ipynb)

In [ ]:
import time
from model.utils.model_io import save_checkpoint
from model.utils.plot import plot_loss_history
from model.utils.masking import create_causal_doc_mask
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from torch.optim.lr_scheduler import LambdaLR
    from torch.utils.data import DataLoader
    import torch.optim as optim
    from model.model_text_only import TextOnlyModel


class PreTrainingModel:
    def __init__(self, cfg: BaseConfig, model: TextOnlyModel,
                 optimizer:optim.AdamW,
                 scheduler:LambdaLR,
                 ):
        """
        Implement the pre-training.
        """
        self.cfg = cfg
        self.model = model

        self.criterion = nn.CrossEntropyLoss(
            # label_smoothing=0.1,  # Used Torch's built-in Label Smoothing.
            ignore_index=cfg.special_tokens["pad_token"]["ID"],
            # reduction="sum",
        )
        self.optimizer= optimizer
        self.scheduler= scheduler

        # Track the training steps. Once it reach `total_training_steps` training ends.
        self.step_counter = 0
        self.loss_history = []
        self.start_time = None  # Track the start time of the training

    def train(self, train_dataloader: DataLoader, continue_from_step=0, overfit=False, print_every:int=50):
        """
        Pre-train the model.

        Args:
            continue_from_step: If continuing training, start from the last step the model was trained on.
            train_dataloader: A dataloader that yields packed batches of tokens.
            overfit: Whether to do a small overfit test.
            print_every: Print every n steps.
        """

        cfg = self.cfg
        self.step_counter = continue_from_step

        # Convert the dataloader stream into an iterator for step-based fetching
        data_iter = iter(train_dataloader)

        if overfit:
            print("\n\nOverfitting the model on a single batch......")
            static_x, static_y = next(data_iter)
            grad_accum_steps = 1  # Temporarily override gradient accumulation to 1 for direct updates, needed for overfitting test
            # Override the 0.0 warmup learning rate, this is because we skip self.scheduler.step() during overfit, so learning rate never increments.
            for param_group in self.optimizer.param_groups:
                param_group["lr"] = cfg.peak_lr
        else:
            grad_accum_steps = cfg.gradient_accumulation_steps
            print("\n" + "#" * 64)
            print(
                f"\nPre-Training Model | Total Training Steps: {cfg.total_training_steps:,} | "
                f"Accumulation Steps: {grad_accum_steps}"
            )
            print("\n" + "#" * 64)

        self.start_time = time.time()
        # TODO add interval checkpoint saving so we dont lose all model info if error.
        # TODO Maybe add a validation loop to validate while training
        while self.step_counter < cfg.total_training_steps:
            # NOTE: If you are training on massive GPU clusters, use HuggingFace's Accelerator for gradient accumulation!
            accumulated_loss = 0.0

            # === Inner loop: Accumulate gradients
            for micro_step in range(grad_accum_steps):
                if overfit:
                    x_ids, y_ids = static_x, static_y
                else:
                    try:
                        # Fetch exactly one micro-batch from the huggingFace stream
                        x_ids, y_ids = next(data_iter)
                    except StopIteration:
                        # Fallback in case a finite dataset exhausts
                        data_iter = iter(train_dataloader)
                        x_ids, y_ids = next(data_iter)

                x_ids = x_ids.to(cfg.device)
                y_ids = y_ids.to(cfg.device)

                # Run micro-batch, and do not call optimizer.step() there!
                loss = self._run_micro_batch(x_ids, y_ids, grad_accum_steps)
                accumulated_loss += loss

            # === Outer loop: The global step
            # Clip gradients to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

            # Apply the accumulated gradients to the weights
            self.optimizer.step()
            if not overfit:
                self.scheduler.step()
            self.optimizer.zero_grad()

            self.loss_history.append(accumulated_loss)
            self.step_counter += 1

            if self.step_counter % print_every == 0:
                print(
                    f"Step [{self.step_counter}/{cfg.total_training_steps}] Loss: {accumulated_loss:.4f}"
                )
            if self.step_counter % 500 == 0 and not overfit:
                # Save a checkpoint every 500 steps
                save_checkpoint(
                    cfg=self.cfg,
                    model=self.model,
                    optimizer=self.optimizer,
                    scheduler=self.scheduler,
                    step_counter=self.step_counter,
                    loss_history=self.loss_history,
                    avg_loss=accumulated_loss,
                )

        # TODO when I create the other posttraining and the training stages
        cfg.training_stage = "pre-trained"

        save_checkpoint(
            cfg=self.cfg,
            model=self.model,
            optimizer=self.optimizer,
            scheduler=self.scheduler,
            step_counter=self.step_counter,
            loss_history=self.loss_history,
            avg_loss=accumulated_loss,
        )
        cfg.save_config_to_json()
        plot_loss_history(cfg=cfg, loss_history=self.loss_history, step_counter=self.step_counter)
        print("Training complete!\n\n")

    def _run_micro_batch(self, x_ids, y_ids, grad_accum_steps):
        """
        Run a single forward and backward pass for a micro-batch.

        Args:
            x_ids: The input token ids.
            y_ids: The output token ids.
        """
        # Set the model to train mode
        self.model.train()

        # Generated the doc and causal mask
        batch_mask = create_causal_doc_mask(cfg=self.cfg, token_ids=x_ids)

        # Forward pass
        logits = self.model(x_ids, batch_mask)

        # Compute the loss
        loss = self.criterion(
            logits.view(-1, self.cfg.vocab_size), y_ids.view(-1)
        )  # Flatten to 2D and 1D for CrossEntropy

        # Scale the loss to account for gradient accumulation
        scaled_loss = loss / grad_accum_steps

        # Backward pass
        scaled_loss.backward()

        # Return the unscaled loss
        return loss.item()

## OVERFIT TEST:

In [5]:
# @i-c
from llama_configs import Llama3_scaled_down
from model.model_text_only import TextOnlyModel
from model.tokenizer import BPETokenizer
from model.utils.data_loader import create_causal_dataloader
from model.utils.data_loader import get_raw_dataset
from model.utils.optimizer_and_scheduler import get_optimizer, get_scheduler

print("\n ---------- OVERFIT TEST ---------- \n")
cfg = Llama3_scaled_down()

# Temporarily override gradient accumulation to 1 for direct updates, needed for overfitting test
cfg.gradient_accumulation_steps = 1
cfg.token_budget = 1 * cfg.global_batch_size_tokens

cfg.CURR_CHPT_DIR = cfg.CHPTS_DIR / "Scaled_down_Llama_3_1_OVERFIT"


tokenizer = BPETokenizer(cfg)
success, trained_tokenizer = tokenizer.load_tokenizer()
if not success:
    trained_tokenizer = tokenizer.train_tokenizer(get_raw_dataset())

model = TextOnlyModel(cfg).to(cfg.device)

optimizer = get_optimizer(cfg, model=model)
scheduler = get_scheduler(cfg, optimizer=optimizer)

trainer = PreTrainingModel(cfg, model=model, optimizer=optimizer, scheduler=scheduler)  

/Users/tonyavis/miniconda3/envs/how_to_create_llm/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



 ---------- OVERFIT TEST ---------- 


Project Root: /Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM

Loading existing trained BPE Tokenizer with vocab size:32000 from: (/Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM/model/saved_models/universal_tokenizer/custom_BPE_tokenizer_vocab_size_32000.json)...


Model initialized with 99,528,704 parameters!




In [6]:
# @i-c:
# Fetch a single batch
print("\n\nFetching a single batch...")
dataloader = create_causal_dataloader(cfg, trained_tokenizer)



Fetching a single batch...


Took about ~4 mins to run the overfit test on my M1 Mac Max with 64GB RAM.

In [7]:
# @i-c: 
trainer.train(dataloader, overfit=True)
print("If the loss approaches 0.0, the model architecture and gradient flow are functional.")



Overfitting the model on a single batch......
Saved Checkpoint to -> /Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM/model/checkpoints/Scaled_down_Llama_3_1_OVERFIT/step_000001.pt
Saved Config to -> /Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM/model/checkpoints/Scaled_down_Llama_3_1_OVERFIT/config.json
Loss plot saved to /Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM/model/checkpoints/Scaled_down_Llama_3_1_OVERFIT/step_000001_training_loss_plot.png
Training complete!


If the loss approaches 0.0, the model architecture and gradient flow are functional.
